In [1]:
from langgraph.graph import StateGraph,START,END
from langchain_groq import ChatGroq
from typing import TypedDict
from dotenv import load_dotenv

e:\langraph\myvenv\Lib\site-packages\langgraph\cache\base\__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [2]:
load_dotenv()

True

In [4]:
model = ChatGroq(
    model="llama-3.1-8b-instant"  # free model
    
)

In [5]:
#create a state
class LLMState(TypedDict):
    qn:str
    ans:str


In [6]:
def llmqa(state:LLMState) ->LLMState:
    #extract the qn from state
    qnn=state['qn']
    #form a prompt
    prompt=f'Answer the following question: {qnn}'
    #ask that qn to llm
    answer=model.invoke(prompt).content
    #update ans in state
    state['ans']=answer
    return state


In [7]:
#create graph
graph=StateGraph(LLMState)

#add nodes
graph.add_node('llm_qa',llmqa)
#add edges
graph.add_edge(START, 'llm_qa')
graph.add_edge('llm_qa', END)
#compile
workflow=graph.compile()

In [8]:
#execute
initial_state={'qn':'What is the capital of France?','ans':''}
final_state=workflow.invoke(initial_state)
print(final_state)

{'qn': 'What is the capital of France?', 'ans': 'The capital of France is Paris.'}
